In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib
import numpy as np
import keras
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']  
plt.rcParams['axes.unicode_minus'] = False 
from keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D, Conv1D, MaxPooling1D
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping
from cnsenti import Sentiment
import jieba as jb
import re

In [3]:
# 0. Loading Train & Test Dataset
df = pd.read_csv(r"E:\【桌面】\HKU\【7012】Social Media\Group Project\情感分类\data\news_data.csv")
df["label"] = 2 - df["label"] # 0-negative, 1-neutral, 2-positive
df

,Unnamed: 0,NewsID,title,label
0,0,63c7e2a3e2394ba391c45c8e591177f7,风清扬:瞧，柳传志们那群惊弓之鸟! - 网友杂谈 - 红歌会网,1
1,1,bafd0522ca0e4c6db074cb2d1279139e,"白人少女被刺伤 拒黑人医生诊治""你太脏了""_国际图文_看看新闻",0
2,2,26af69393849430aa18ed0f1b982160a,惊心!2岁女童掉进厦门火车站站台，火车即将发车...,0
3,3,f8158e2a0ce04f11999f04b4beae59b3,靖远县城区一违法建筑被强制拆除,2
4,4,89dfbe8b90e84819a96ab94a9b52ff19,干货 | 一周新闻知多少,1
...,...,...,...,...
15339,995,105241382,A股集体反弹！沪指收复3400点 创指涨逾3%,2
15340,996,105250119,紫光股份：旗下新华三集团中标光大银行ARM平台服务器项目,1
15341,997,105148619,欢喜传媒(01003.HK)：欢喜首映与华为视频战略合作升级,2
15342,998,105603837,洪兴股份：拟与广州市白云区人民政府签订合作框架协议,1


In [4]:
# 1.Random Guess：Accuracy = 48.77%
label_distribution = df["label"].value_counts()
print(max(label_distribution) / sum(label_distribution))

0.4876824817518248


In [5]:
# 2.3 Use statistics num_pos_word, num_neg_word, length as input features to predict sentiment score
#(1)Load pos & neg word list to calculate statistics
from cnsenti import Sentiment
senti = Sentiment(pos = r'E:\【桌面】\HKU\【7012】Social Media\Group Project\情感分类\data\dict\pos.txt',
                  neg = r'E:\【桌面】\HKU\【7012】Social Media\Group Project\情感分类\data\dict\neg.txt',
                  encoding = 'utf-8') 

# (2)利用cnsenti，统计每篇新闻的title的长度，pos个数，neg个数
df["num_pos_word"] = 0
df["num_neg_word"] = 0
df["length"]       = 0
for index in range(len(df)):
    test_text = str(df.loc[[index],["title"]].values[0][0])
    result    = senti.sentiment_count(test_text)
    df.loc[[index],["length"]]       = result["words"]
    df.loc[[index],["num_pos_word"]] = result["pos"]
    df.loc[[index],["num_neg_word"]] = result["neg"]

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\chens\AppData\Local\Temp\jieba.cache
Loading model cost 0.590 seconds.
Prefix dict has been built successfully.


In [12]:
# 2.1 Naive Bayes：Accuracy = 55.69%
import sklearn
from sklearn.naive_bayes import GaussianNB
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split

X = df[["num_pos_word","num_neg_word","length"]].values
y = df[["label"]].values.reshape(1,-1)[0]

X_train, X_test, y_train, y_test = train_test_split(X,y, shuffle = True, train_size = 0.8, test_size = 0.2)
classifier = GaussianNB()
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)
print(classifier.score(X_test, y_test))

0.5568589116976214


In [13]:
# 2.2 Decision Tree：Accuracy = 59.53% 
#     Random Forest：Accuracy = 58.91%
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

DT_params_list  = range(1,21,1)
DT_perform_list = []
for i in DT_params_list:
    DT_classifier = DecisionTreeClassifier(max_depth = i)
    DT_classifier.fit(X_train, y_train)
    DT_perform_list.append(DT_classifier.score(X_test, y_test))

RF_params_list = range(10, 210, 10)
RF_perform_list = []
for j in RF_params_list:
    RF_classifier = RandomForestClassifier(n_estimators = j)
    RF_classifier.fit(X_train, y_train)
    RF_perform_list.append(RF_classifier.score(X_test, y_test))

print("Best Acc of Decision Tree:", max(DT_perform_list))
print("Best Acc of Random Forest:", max(RF_perform_list))

Best Acc of Decision Tree: 0.5953079178885631
Best Acc of Random Forest: 0.5891169762137504


In [18]:
# 2.3 MLP：Accuracy = 59.65%
import keras

X = df[["num_pos_word","num_neg_word","length"]].values
y = pd.get_dummies(df['label']).values

X_train_val = X[int(len(X)*0.2)::]
y_train_val = y[int(len(X)*0.2)::]
X_test      = X[:int(len(X)*0.2):]
y_test      = y[:int(len(X)*0.2):]

input_layer  = keras.layers.Input(shape = [3])
hidden_1     = keras.layers.Dense(32,activation = "relu", kernel_regularizer = keras.regularizers.l2(0.01))(input_layer)
dropout_1    = keras.layers.Dropout(0.2)(hidden_1)
hidden_2     = keras.layers.Dense(32,activation = "relu", kernel_regularizer = keras.regularizers.l2(0.01))(dropout_1)
dropout_2    = keras.layers.Dropout(0.2)(hidden_2)
output_layer = keras.layers.Dense(3,activation = "softmax")(dropout_2)

model = keras.Model(inputs  = [input_layer],
                    outputs = [output_layer])

model.compile(optimizer = keras.optimizers.RMSprop(learning_rate = 0.01),
              loss      = "categorical_crossentropy",
              metrics   = ["accuracy"])

callbacks_list = [keras.callbacks.EarlyStopping(monitor = "val_accuracy", patience = 1),
                  keras.callbacks.ModelCheckpoint(filepath = "my_model.h5", monitor = "val_loss", save_best_only = True)]
history = model.fit(X_train_val, y_train_val,
                    validation_split = 0.2,
                    epochs = 4,
                    batch_size = 128,
                    callbacks = callbacks_list
                    )

Epoch 1/4
77/77 [==============================] - 1s 7ms/step - loss: 1.1530 - accuracy: 0.4989 - val_loss: 0.9371 - val_accuracy: 0.5896
Epoch 2/4
77/77 [==============================] - 0s 2ms/step - loss: 0.9752 - accuracy: 0.5503 - val_loss: 0.9173 - val_accuracy: 0.5998
Epoch 3/4
77/77 [==============================] - 0s 3ms/step - loss: 0.9417 - accuracy: 0.5599 - val_loss: 0.8255 - val_accuracy: 0.6405
Epoch 4/4
77/77 [==============================] - 0s 2ms/step - loss: 0.9327 - accuracy: 0.5662 - val_loss: 0.8457 - val_accuracy: 0.6152


In [19]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Acc:",test_acc)

96/96 [==============================] - 0s 1ms/step - loss: 0.9092 - accuracy: 0.5965
Test Acc: 0.5964797735214233


In [20]:
# 3. Using TF-IDF Model to transfer word sequences into vectors
#    Then PCA is conducted to realize dimension reduction
#    After PCA, rerun the same models

# text_normalize
def remove_punctuation(line):
    line = str(line)
    if line.strip()=='':
        return ''
    rule = re.compile(u"[^a-zA-Z0-9\u4E00-\u9FA5]")
    line = rule.sub('',line)
    return line

def stopwordslist(filepath):  
    stopwords = [line.strip() for line in open(filepath, 'r', encoding='utf-8').readlines()]  
    return stopwords  

# filter stopwords 
stopwords = stopwordslist(r"E:\【桌面】\HKU\【7012】Social Media\Group Project\情感分类\data\stopwords_2.txt")

In [21]:
df['clean_review'] = df['title'].apply(remove_punctuation)
df['cut_review']   = df['clean_review'].apply(lambda x: " ".join([w for w in list(jb.cut(x)) if w not in stopwords]))

In [22]:
norm_corpus = df['cut_review'].values
from sklearn.feature_extraction.text import TfidfVectorizer
tv = TfidfVectorizer(min_df = 0.0009, max_df = 0.001, norm = "l2", use_idf = True, smooth_idf = True)
tv_matrix = tv.fit_transform(norm_corpus)
tv_matrix = tv_matrix.toarray()
vocab = tv.get_feature_names_out()

X = tv_matrix
y = df[["label"]].values.reshape(1,-1)[0]
# y = pd.get_dummies(df['label']).values

X_train_val = X[int(len(X)*0.2)::]
y_train_val = y[int(len(X)*0.2)::]
X_test      = X[:int(len(X)*0.2):]
y_test      = y[:int(len(X)*0.2):]

# PCA
from sklearn.decomposition import PCA
pca = PCA(n_components = 0.6)
pca.fit(X_train_val)
X_train_val_new = pca.transform(X_train_val)
X_test_new      = pca.transform(X_test)

In [23]:
# 3.1 Naive Beyas：Accuracy = 40.84%
classifier = GaussianNB()
classifier.fit(X_train_val_new, y_train_val)

y_pred = classifier.predict(X_test_new)
print(classifier.score(X_test_new, y_test))

0.40840938722294656


In [24]:
# 3.2 Decision Tree：Accuracy = 51.30% 
#     Random Forest：Accuracy = 51.17%
DT_params_list  = range(1,21,1)
DT_perform_list = []
for i in DT_params_list:
    DT_classifier = DecisionTreeClassifier(max_depth = i)
    DT_classifier.fit(X_train_val_new, y_train_val)
    DT_perform_list.append(DT_classifier.score(X_test_new, y_test))

RF_params_list = range(10, 210, 10)
RF_perform_list = []
for j in RF_params_list:
    RF_classifier = RandomForestClassifier(n_estimators = j)
    RF_classifier.fit(X_train_val_new, y_train_val)
    RF_perform_list.append(RF_classifier.score(X_test_new, y_test))

print("Best Acc of Decision Tree:", max(DT_perform_list))
print("Best Acc of Random Forest:", max(RF_perform_list))

Best Acc of Decision Tree: 0.5130378096479792
Best Acc of Random Forest: 0.5117340286831812


In [25]:
# 3.3 MLP：Accuracy = 48.44%
y_train_val = pd.get_dummies(y_train_val).values
y_test      = pd.get_dummies(y_test).values

input_layer  = keras.layers.Input(shape = [X_train_val_new.shape[1]])
hidden_1     = keras.layers.Dense(32,activation = "relu", kernel_regularizer = keras.regularizers.l2(0.01))(input_layer)
dropout_1    = keras.layers.Dropout(0.2)(hidden_1)
hidden_2     = keras.layers.Dense(32,activation = "relu", kernel_regularizer = keras.regularizers.l2(0.01))(dropout_1)
dropout_2    = keras.layers.Dropout(0.2)(hidden_2)
output_layer = keras.layers.Dense(3,activation = "softmax")(dropout_2)

model = keras.Model(inputs  = [input_layer],
                    outputs = [output_layer])

model.compile(optimizer = keras.optimizers.RMSprop(learning_rate = 0.01),
              loss      = "categorical_crossentropy",
              metrics   = ["accuracy"])

callbacks_list = [keras.callbacks.EarlyStopping(monitor = "val_accuracy", patience = 1),
                  keras.callbacks.ModelCheckpoint(filepath = "my_model.h5", monitor = "val_loss", save_best_only = True)]
history = model.fit(X_train_val_new, y_train_val,
                    validation_split = 0.2,
                    epochs = 10,
                    batch_size = 128,
                    callbacks = callbacks_list
                    )

Epoch 1/10
77/77 [==============================] - 1s 5ms/step - loss: 1.1099 - accuracy: 0.4509 - val_loss: 0.9975 - val_accuracy: 0.6234
Epoch 2/10
77/77 [==============================] - 0s 3ms/step - loss: 1.0637 - accuracy: 0.4548 - val_loss: 0.9920 - val_accuracy: 0.6234


In [26]:
test_loss, test_acc = model.evaluate(X_test_new, y_test)
print("Test Acc:",test_acc)

96/96 [==============================] - 0s 1ms/step - loss: 1.0154 - accuracy: 0.4844
Test Acc: 0.4843546152114868
